# RL Jammer DSP acceleration prototype (`RL_Jammer_testing.ipynb`)

This notebook **implements** (not just documents) DSP-chain accelerations:
- vectorized/batched STFT feature extraction,
- cached DSP constants (window/frequency grid),
- decode scheduling for reward computation,
- batched decode calls through `rx_command_iq_broadcast`,
- resample short-circuit when sample rates match.


In [1]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, List, Sequence, Tuple
import numpy as np
import math
import torch
from pathlib import Path
import os
import logging

from mpmath.libmp.libelefun import log_int_cache

import advanced_link_skdsp_v7_robust as link7
import score_iq_decode as scorer
import copy
from accelerated_training_utils import repeat_to_length_mod

if not hasattr(np, "bool8"):
    np.bool8 = np.bool_

from torch.utils.tensorboard import SummaryWriter

from accelerated_training_utils import (JammerVecEnv,
                                        jammer_controller_batch)
from tx_controller_tone_pulse_stft_varlen_9 import ActorCritic, TonePulseTXControlNetVarLen

logger = logging.getLogger(__name__)

from accelerated_training_utils import (
    precompute_training_cache,
    create_cached_dataloader,
    )

import gc

def clean():
    while gc.collect() >0:
        gc.collect()
    return


In [2]:
import gc
import torch

# del tensor_or_model_or_batch
gc.collect()
torch.cuda.empty_cache()

In [3]:
clean()

In [4]:

print(torch.cuda.memory_summary())
print(torch.cuda.memory_allocated() / 1024**2, "MiB allocated")
print(torch.cuda.memory_reserved() / 1024**2, "MiB reserved")

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |      0 B   |      0 B   |      0 B   |      0 B   |
|       from large pool |      0 B   |      0 B   |      0 B   |      0 B   |
|       from small pool |      0 B   |      0 B   |      0 B   |      0 B   |
|---------------------------------------------------------------------------|
| Active memory         |      0 B   |      0 B   |      0 B   |      0 B   |
|       from large pool |      0 B   |      0 B   |      0 B   |

In [5]:
def detach_obs(obs):
    return {
        "stft_feature_list": [x.detach() for x in obs["stft_feature_list"]],
        **({
            "scalar_side": obs["scalar_side"].detach()
        } if "scalar_side" in obs else {}),
    }

In [6]:
_DSP_CACHE: Dict[Tuple[str, int, int, float], Dict[str, torch.Tensor]] = {}


def _get_stft_constants(device: torch.device,
                        nfft: int,
                        nperseg: int,
                        sample_rate_hz: float) -> Dict[str, torch.Tensor]:
    key = (str(device), int(nfft), int(nperseg), float(sample_rate_hz))
    cached = _DSP_CACHE.get(key)
    if cached is not None:
        return cached

    window = torch.hann_window(nperseg, dtype=torch.float32, device=device)
    fgrid = torch.fft.fftshift(
        torch.fft.fftfreq(nfft, d=1.0 / max(float(sample_rate_hz), 1.0)).to(device)
    ).to(torch.float32)
    out = {"window": window, "fgrid": fgrid}
    _DSP_CACHE[key] = out
    return out


def preprocess_batched_iq_to_stft_feature_fast(
    iq_batch: torch.Tensor,
    sample_rate_hz: float,
    nperseg: int = 128,
    noverlap: int = 96,
    nfft: int = 128,
    out_channels: int = 14,
) -> Dict[str, torch.Tensor]:
    """Vectorized STFT preprocessing for a complex IQ batch [B, T]."""
    if not torch.is_tensor(iq_batch):
        iq_batch = torch.as_tensor(iq_batch)
    if iq_batch.ndim != 2:
        raise ValueError(f"iq_batch must be [B, T], got {tuple(iq_batch.shape)}")

    x = iq_batch.to(dtype=torch.complex64)
    # Cached/test IQ rows can occasionally contain non-finite samples (or very
    # large outliers after resampling).  Sanitize before reductions/STFT so a
    # single bad row cannot turn the whole SAC action distribution into NaNs.
    x = torch.complex(
        torch.clamp(torch.nan_to_num(x.real, nan=0.0, posinf=0.0, neginf=0.0), min=-1.0e6, max=1.0e6),
        torch.clamp(torch.nan_to_num(x.imag, nan=0.0, posinf=0.0, neginf=0.0), min=-1.0e6, max=1.0e6),
    )
    if x.shape[1] < 8:
        x = torch.nn.functional.pad(x, (0, 8 - x.shape[1]))

    x = x - x.mean(dim=1, keepdim=True)
    scale = torch.median(torch.abs(x), dim=1, keepdim=True).values
    x = x / (scale + 1e-6)

    nperseg_eff = int(min(nperseg, max(8, x.shape[1])))
    noverlap_eff = int(min(noverlap, nperseg_eff - 1))
    hop_length = max(1, nperseg_eff - noverlap_eff)

    const = _get_stft_constants(device=x.device, nfft=nfft, nperseg=nperseg_eff, sample_rate_hz=sample_rate_hz)

    z = torch.stft(
        x,
        n_fft=nfft,
        hop_length=hop_length,
        win_length=nperseg_eff,
        window=const["window"],
        center=False,
        return_complex=True,
    )
    z = torch.fft.fftshift(z, dim=1)

    mag = torch.log1p(torch.abs(z)).to(torch.float32)
    phase = torch.angle(z).to(torch.float32)
    real = z.real.to(torch.float32)
    imag = z.imag.to(torch.float32)
    power = (torch.abs(z) ** 2).to(torch.float32)
    power_log = torch.log1p(power)

    f = const["fgrid"][None, :, None]
    denom = power.sum(dim=1, keepdim=True) + 1e-12
    centroid = (f * power).sum(dim=1, keepdim=True) / denom
    centroid_plane = centroid.repeat(1, mag.shape[1], 1) / max(float(sample_rate_hz), 1.0)

    spread = torch.sqrt((((f - centroid) ** 2) * power).sum(dim=1, keepdim=True) / denom + 1e-12)
    spread_plane = (spread / max(float(sample_rate_hz), 1.0)).repeat(1, mag.shape[1], 1)

    flatness = torch.exp(torch.mean(torch.log(torch.clamp(power, min=1e-12)), dim=1, keepdim=True))
    flatness = flatness / (power.mean(dim=1, keepdim=True) + 1e-12)
    flatness_plane = flatness.repeat(1, mag.shape[1], 1)

    frame_power = power.mean(dim=1, keepdim=True)
    frame_power_norm = frame_power / (frame_power.mean(dim=2, keepdim=True) + 1e-12)
    frame_power_plane = frame_power_norm.repeat(1, mag.shape[1], 1)

    delta_t_mag = torch.diff(mag, dim=2, prepend=mag[:, :, :1])
    delta_f_mag = torch.diff(mag, dim=1, prepend=mag[:, :1, :])
    delta_t_power = torch.diff(power_log, dim=2, prepend=power_log[:, :, :1])
    delta_t_phase = torch.diff(phase, dim=2, prepend=phase[:, :, :1])

    sr_plane = torch.full_like(mag, torch.log10(torch.tensor(max(float(sample_rate_hz), 1.0), device=mag.device)) / 10.0)

    feat = torch.stack(
        [
            mag,
            phase,
            centroid_plane,
            sr_plane,
            real,
            imag,
            power_log,
            delta_t_mag,
            delta_f_mag,
            delta_t_phase,
            delta_t_power,
            spread_plane,
            flatness_plane,
            frame_power_plane,
        ][:out_channels],
        dim=1,
    )

    peak_bin = torch.argmax(power.mean(dim=2), dim=1)
    peak_hz = const["fgrid"][peak_bin]
    rx_power = (x.abs() ** 2).mean(dim=1)

    return {
        "feature": torch.clamp(torch.nan_to_num(feat.to(torch.float32), nan=0.0, posinf=1.0e6, neginf=-1.0e6), min=-1.0e6, max=1.0e6),
        "rx_power": rx_power.to(torch.float32),
        "peak_hz": peak_hz.to(torch.float32),
        "lengths": torch.full((x.shape[0],), float(x.shape[1]), device=x.device, dtype=torch.float32),
    }


In [7]:
def stack_iq_views_to_device(
    samples: Sequence[Dict[str, Any]],
    *,
    device: str,
    dtype: torch.dtype = torch.complex64,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Stack active sample IQ views and transfer each view to the target device once."""
    dev = torch.device(device)
    return tuple(
        torch.stack([s[name] for s in samples], dim=0).to(
            device=dev,
            dtype=dtype,
            non_blocking=True,
        )
        for name in ("iq1", "iq2", "iq3")
    )


def build_stft_observation_fast(
    *,
    iq1: torch.Tensor,
    iq2: torch.Tensor,
    iq3: torch.Tensor,
    intake_sample_rate_hz: float,
    device: str = "cpu",
) -> Dict[str, List[torch.Tensor]]:
    # Move raw IQ to the target device before STFT feature extraction.  This keeps
    # the CPU->GPU boundary at the smaller IQ tensors instead of the expanded
    # multi-channel STFT feature tensor, and it lets CUDA run the STFT pipeline.
    dev = torch.device(device)
    view_batch = torch.cat([iq1, iq2, iq3], dim=0).to(
        device=dev,
        dtype=torch.complex64,
        non_blocking=True,
    )
    processed = preprocess_batched_iq_to_stft_feature_fast(view_batch, intake_sample_rate_hz)
    features = processed["feature"].to(dtype=torch.float32)
    f1, f2, f3 = features.chunk(3, dim=0)
    return {
        "stft_feature_list": [f1, f2, f3]
    }


In [8]:
@dataclass
class RewardScheduleConfig:
    decode_every_n_steps: int = 4
    proxy_reward_scale: float = 0.01


def _batched_decode(iq_batch: torch.Tensor, meta_list: Sequence[Dict[str, Any]]) -> List[Dict[str, Any]]:
    if hasattr(link7, "rx_command_iq_broadcast"):
        return link7.rx_command_iq_broadcast(iq_batch, list(meta_list))
    return [link7.rx_command_iq(iq_batch[i], meta_list[i]) for i in range(iq_batch.shape[0])]


def compute_rewards_scheduled_batched(
    *,
    jam_batch: Sequence[Dict[str, Any]],
    samples: Sequence[Dict[str, Any]],
    jammer_sampling_freq: float,
    device: str,
    step_idx: int,
    sched: RewardScheduleConfig,
) -> Tuple[torch.Tensor, int, int]:
    """Batched reward computation with decode scheduling + resample short-circuit."""
    dev = torch.device(device)
    do_decode = (step_idx % max(1, int(sched.decode_every_n_steps)) == 0)

    jammed_rows: List[torch.Tensor] = []
    metas: List[Dict[str, Any]] = []
    proxy_rewards: List[torch.Tensor] = []

    for jam_item, sample in zip(jam_batch, samples):
        tx_iq = torch.as_tensor(jam_item["tx_iq"], dtype=torch.complex64, device=dev)
        whole_iq = sample["whole_iq"]
        whole_meta = sample["whole_meta"]
        sr_out = float(whole_meta["sample_rate_hz"])

        if abs(float(jammer_sampling_freq) - sr_out) < 1e-6:
            tx_rs = tx_iq
        else:
            tx_rs = link7.resample_iq(tx_iq, fs_in_hz=float(jammer_sampling_freq), fs_out_hz=sr_out)

        tx_rs = repeat_to_length_mod(tx_rs, int(whole_iq.shape[0]))
        tx_rs = torch.as_tensor(tx_rs[: whole_iq.shape[0]], dtype=torch.complex64, device=dev)
        proxy_rewards.append((-float(sched.proxy_reward_scale) * torch.mean(torch.abs(tx_rs) ** 2)).to(dtype=torch.float32))

        if do_decode:
            jammed_rows.append(torch.as_tensor(whole_iq, dtype=torch.complex64, device=dev) + tx_rs)
            metas.append(whole_meta)

    if not proxy_rewards:
        return torch.zeros(0, dtype=torch.float32, device=dev), 0, 0

    if not do_decode:
        return torch.stack(proxy_rewards), 0, 0

    max_len = max(int(x.numel()) for x in jammed_rows)
    jammed_pad = [torch.nn.functional.pad(x, (0, max_len - int(x.numel()))) for x in jammed_rows]
    jammed_batch = torch.stack(jammed_pad, dim=0)

    results = _batched_decode(jammed_batch, metas)

    score_values: List[float] = []
    decoded = 0
    total = 0
    for r, m in zip(results, metas):
        total += 1
        s = float(scorer.score_decode(r, m))
        if s < 1.0:
            decoded += 1
        score_values.append(s)

    return torch.as_tensor(score_values, dtype=torch.float32, device=dev), decoded, total


In [9]:
def _resolve_steps_per_epoch_for_notebook(env: JammerVecEnv, rollout_steps: int = 0) -> int:
    if int(rollout_steps) > 0:
        return int(rollout_steps)
    if hasattr(env, "_source_batches_per_epoch") and int(getattr(env, "_source_batches_per_epoch", 0)) > 0:
        return int(env._source_batches_per_epoch)
    sample_count = len(getattr(env, "samples", []))
    return max(1, int(np.ceil(sample_count / max(1, int(env.num_envs)))))


@torch.no_grad()
def _evaluate_split_metrics_batched_dsp(
    policy: torch.nn.Module,
    env: JammerVecEnv,
    split: str,
    device: str,
    value_coef: float,
    reward_sched: RewardScheduleConfig,
    output_len: int
) -> Tuple[float, float]:
    split_samples = getattr(env, "test_samples", None) if split == "test" else getattr(env, "samples", None)
    if not split_samples:
        return float("inf"), 0.0

    logp_buf: List[torch.Tensor] = []
    rew_buf: List[torch.Tensor] = []
    val_buf: List[torch.Tensor] = []
    decodes = 0
    decode_total = 0
    eval_step_idx = 0

    original_mode = env.mode
    original_active = list(getattr(env, "_active", []))
    original_step_count = int(getattr(env, "_step_count", 0))

    try:
        env.set_mode(split)
        obs = env.reset()
        eval_steps = _resolve_steps_per_epoch_for_notebook(env, 0)

        for _ in range(eval_steps):
            model_obs = build_stft_observation_fast(
                iq1=obs["iq1"],
                iq2=obs["iq2"],
                iq3=obs["iq3"],
                intake_sample_rate_hz=float(env.jammer_sampling_freq),
                device=device,
            )
            action_t, value_t, logp_t = policy.get_action_value_logp(model_obs)
            actions = action_t.squeeze()
            if env.num_envs == 1 and isinstance(actions, torch.Tensor) and actions.ndim >= 1:
                action_batch = [actions]
            else:
                action_batch = actions

            if not getattr(env, "_active", None):
                env._active = env._next_samples()
            active_samples = list(env._active)

            jam_batch = jammer_controller_batch(
                model=env.model,
                samples=active_samples,
                actions=action_batch,
                default_output_len = env.default_output_len,
                jammer_sampling_freq=env.jammer_sampling_freq,
                user_peak_power_fraction=env.user_peak_power_fraction,
                device=env.device,
            )

            rewards_t, decode, total = compute_rewards_scheduled_batched(
                jam_batch=jam_batch,
                samples=active_samples,
                jammer_sampling_freq=float(env.jammer_sampling_freq),
                device=device,
                step_idx=eval_step_idx,
                sched=reward_sched,
            )
            eval_step_idx += 1
            decodes += int(decode)
            decode_total += int(total)

            env._step_count += 1
            done = env._step_count >= env.max_steps
            if done:
                env._active = env._next_samples()
                env._step_count = 0
            obs = env._obs_from_samples(env._active)

            logp_buf.append(logp_t)
            val_buf.append(value_t)
            rew_buf.append(rewards_t.to(device=device, dtype=torch.float32))

        returns = torch.stack(rew_buf, dim=0)
        values = torch.stack(val_buf, dim=0)
        logps = torch.stack(logp_buf, dim=0)
        if logps.ndim > returns.ndim:
            logps = logps.squeeze(-1)
        if values.ndim > returns.ndim:
            values = values.squeeze(-1)

        adv =  returns #- values.detach()
        # adv = (adv - adv.mean()) / (returns.std(unbiased=False) + 1)
        policy_loss = -(logps * adv.detach()).mean()
        # critic_loss = torch.nn.functional.mse_loss(values, returns)
        loss = policy_loss #+ value_coef * critic_loss
        # print(f" test logps.mean() : {logps.mean()}")
        # print(f" test adv.mean() : {adv.mean()}")

        decode_pct = (100.0 * decodes / decode_total) if decode_total else 0.0

        return float(loss.item()), float(decode_pct)
    finally:
        env.set_mode(original_mode)
        env._active = original_active
        env._step_count = original_step_count




In [10]:
class SACReplayBuffer:
    """Preallocated tensor ring buffer for one-step SAC/contextual-bandit transitions.

    The first add() lazily allocates contiguous tensors from the observed shapes.
    Keeping storage tensorized avoids per-row Python tuples, random.sample, many
    torch.cat calls, and repeated CPU/GPU round-trips during sample().  The
    current jammer rollout treats every transition as terminal, so the buffer
    stores only (obs, action, reward) and the SAC target is simply y = reward.
    """

    def __init__(self, capacity: int, device: str):
        self.capacity = int(capacity)
        self.device = torch.device(device)
        self.pos = 0
        self.size = 0
        self.obs_storage = None
        self.action_storage = None
        self.reward_storage = None
        self.scalar_storage = None

    def __len__(self):
        return self.size

    def _allocate(self, obs, action, reward):
        self.obs_storage = [
            torch.empty((self.capacity, *x.shape[1:]), dtype=x.dtype, device=self.device)
            for x in obs["stft_feature_list"]
        ]
        self.action_storage = torch.empty((self.capacity, *action.shape[1:]), dtype=action.dtype, device=self.device)
        self.reward_storage = torch.empty((self.capacity, *reward.shape[1:]), dtype=reward.dtype, device=self.device)
        if "scalar_side" in obs:
            self.scalar_storage = torch.empty((self.capacity, *obs["scalar_side"].shape[1:]), dtype=obs["scalar_side"].dtype, device=self.device)

    def _copy_rows(self, storage, values, dst_start: int, src_start: int, count: int):
        if count <= 0:
            return
        src = values[src_start:src_start + count]
        if src.device != self.device:
            src = src.to(self.device, non_blocking=True)
        storage[dst_start:dst_start + count].copy_(src)

    def add(self, obs, action, reward, next_obs=None, done=None):
        # ``next_obs`` and ``done`` are accepted for backwards-compatible call
        # sites but intentionally ignored by the one-step terminal update.
        batch_size = int(action.shape[0])
        if batch_size <= 0:
            return
        if self.obs_storage is None:
            self._allocate(obs, action, reward)

        if batch_size > self.capacity:
            start = batch_size - self.capacity
            obs = {"stft_feature_list": [x[start:] for x in obs["stft_feature_list"]], **({"scalar_side": obs["scalar_side"][start:]} if "scalar_side" in obs else {})}
            action = action[start:]
            reward = reward[start:]
            batch_size = self.capacity

        first = min(batch_size, self.capacity - self.pos)
        second = batch_size - first
        chunks = ((self.pos, 0, first), (0, first, second))

        for dst_start, src_start, count in chunks:
            for dst, src in zip(self.obs_storage, obs["stft_feature_list"]):
                self._copy_rows(dst, src.detach(), dst_start, src_start, count)
            self._copy_rows(self.action_storage, action.detach(), dst_start, src_start, count)
            self._copy_rows(self.reward_storage, reward.detach(), dst_start, src_start, count)
            if self.scalar_storage is not None and "scalar_side" in obs:
                self._copy_rows(self.scalar_storage, obs["scalar_side"].detach(), dst_start, src_start, count)

        self.pos = (self.pos + batch_size) % self.capacity
        self.size = min(self.capacity, self.size + batch_size)

    def sample(self, batch_size: int):
        if self.size <= 0:
            raise ValueError("cannot sample from an empty replay buffer")
        idx = torch.randint(self.size, (int(batch_size),), device=self.device)
        s = {"stft_feature_list": [x.index_select(0, idx) for x in self.obs_storage]}
        if self.scalar_storage is not None:
            s["scalar_side"] = self.scalar_storage.index_select(0, idx)
        a = self.action_storage.index_select(0, idx)
        r = self.reward_storage.index_select(0, idx)
        return s, a, r


In [11]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# epochs = 250
# batch_size = 250
jammer_sampling_freq = 2e9

train_path_dat = Path("C:/Users/theon/Jamming/train_set_00/dataset")
test_path_dat = Path("C:/Users/theon/Jamming/test_set_00/dataset")
checkpoint_dir = Path("C:/Users/theon/Jamming/checkpoints_accelerated")
checkpoint_dir.mkdir(parents=True, exist_ok=True)

# 1) One-time deterministic precompute/cache of whole_iq + resampled iq1/iq2/iq3.
# train_cache_dir = checkpoint_dir / "train_cache"
# test_cache_dir = checkpoint_dir / "test_cache"

train_cache_dir = checkpoint_dir / "train_cache_01"
test_cache_dir = checkpoint_dir / "test_cache_01"

In [12]:
# 2) DataLoader path with worker prefetch + pinned memory.
num_workers = 0 if os.name == "nt" else 4
pin_memory = (device == "cuda")

In [13]:
# max_numeric_suffix=250,
precompute_training_cache(train_path_dat, train_cache_dir, jammer_sampling_freq, section_len=40_000, overwrite=True)
precompute_training_cache(test_path_dat, test_cache_dir, jammer_sampling_freq, section_len=40_000, overwrite=True)

# # max_numeric_suffix=250,
# precompute_training_cache(train_path_dat, train_cache_dir, jammer_sampling_freq, max_numeric_suffix=500,section_len=1024, overwrite=False)
# precompute_training_cache(test_path_dat, test_cache_dir, jammer_sampling_freq, max_numeric_suffix=250,section_len=1024, overwrite=False)


C:\Users\theon\.conda\envs\jammer\Lib\site-packages\torch\_utils.py:260: UserWarning: ComplexHalf support is experimental and many operators don't support it yet. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\EmptyTensor.cpp:55.)
  t = torch.empty(


[WindowsPath('C:/Users/theon/Jamming/checkpoints_accelerated/test_cache_01/sample_000000.pt'),
 WindowsPath('C:/Users/theon/Jamming/checkpoints_accelerated/test_cache_01/sample_000001.pt'),
 WindowsPath('C:/Users/theon/Jamming/checkpoints_accelerated/test_cache_01/sample_000002.pt'),
 WindowsPath('C:/Users/theon/Jamming/checkpoints_accelerated/test_cache_01/sample_000003.pt'),
 WindowsPath('C:/Users/theon/Jamming/checkpoints_accelerated/test_cache_01/sample_000004.pt'),
 WindowsPath('C:/Users/theon/Jamming/checkpoints_accelerated/test_cache_01/sample_000005.pt'),
 WindowsPath('C:/Users/theon/Jamming/checkpoints_accelerated/test_cache_01/sample_000006.pt'),
 WindowsPath('C:/Users/theon/Jamming/checkpoints_accelerated/test_cache_01/sample_000007.pt'),
 WindowsPath('C:/Users/theon/Jamming/checkpoints_accelerated/test_cache_01/sample_000008.pt'),
 WindowsPath('C:/Users/theon/Jamming/checkpoints_accelerated/test_cache_01/sample_000009.pt'),
 WindowsPath('C:/Users/theon/Jamming/checkpoints_a

In [14]:


batch_size = 2
train_loader = create_cached_dataloader(
    train_cache_dir,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=pin_memory,
    prefetch_factor=2,
    persistent_workers=(num_workers > 0),
)
test_loader = create_cached_dataloader(
    test_cache_dir,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=pin_memory,
    prefetch_factor=2,
    persistent_workers=(num_workers > 0),
)



In [15]:
@dataclass
class PPOConfig:
    rollout_steps: int = 0 #28
    updates: int = 60
    epochs: int = 200
    gamma: float = 0.99
    lr: float = 5e-4
    value_coef: float = 0.5
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    tensorboard_log_dir: str = "runs/"
    checkpoint_dir: str = "checkpoints_rl/"
    checkpoint_name: str = "Best_Models/"

In [16]:
class SACActorVarLen(torch.nn.Module):
    """SAC actor with autoregressive mixture sampling for pulse-phase slots."""

    def __init__(self, action_dim: int, in_ch: int = 14, base_ch: int = 24, n_scalar_features: int = 6, max_tones: int = 128, max_pulses: int = 128, pulse_phase_ar_components: int = 4):
        super().__init__()
        self.backbone = TonePulseTXControlNetVarLen(
            in_ch=in_ch,
            base_ch=base_ch,
            n_scalar_features=n_scalar_features,
            max_tones=max_tones,
            max_pulses=max_pulses,
            pulse_phase_ar_components=pulse_phase_ar_components,
        )
        self.max_tones = int(max_tones)
        self.max_pulses = int(max_pulses)
        self.action_dim = int(action_dim)
        self.mu_head = torch.nn.Linear(96, self.action_dim)
        self.log_std_head = torch.nn.Linear(96, self.action_dim)

    def _sanitize_stft_list(self, stft_list: List[torch.Tensor]) -> List[torch.Tensor]:
        return [
            torch.clamp(
                torch.nan_to_num(x, nan=0.0, posinf=1.0e6, neginf=-1.0e6),
                min=-1.0e6,
                max=1.0e6,
            )
            for x in stft_list
        ]

    def _fused(self, obs: Dict[str, torch.Tensor]) -> torch.Tensor:
        stft_list = self._sanitize_stft_list(obs["stft_feature_list"])
        ref = stft_list[0]
        n_scalar = self.backbone.scalar_proj[0].in_features
        scalar_side = obs.get("scalar_side", torch.zeros(ref.shape[0], n_scalar, device=ref.device, dtype=ref.dtype))
        emb = [self.backbone.encoder(x) for x in stft_list]
        z_stft = torch.stack(emb, dim=0).mean(dim=0)
        fused = self.backbone.fusion(torch.cat([z_stft, self.backbone.scalar_proj(scalar_side)], dim=-1))
        return torch.nan_to_num(fused, nan=0.0, posinf=1.0e6, neginf=-1.0e6)

    def _pulse_phase_action_slice(self) -> slice:
        start = 5 + (4 * self.max_tones) + 1
        return slice(start, start + self.max_pulses)

    def sample(self, obs: Dict[str, torch.Tensor]) -> Tuple[torch.Tensor, torch.Tensor]:
        z = self._fused(obs)
        mu = torch.clamp(torch.nan_to_num(self.mu_head(z), nan=0.0, posinf=20.0, neginf=-20.0), min=-20.0, max=20.0)
        log_std = torch.clamp(torch.nan_to_num(self.log_std_head(z), nan=-5.0, posinf=2.0, neginf=-5.0), -5.0, 2.0)
        std = torch.exp(log_std)
        dist = torch.distributions.Normal(mu, std)
        u = dist.rsample()
        squashed_a = torch.tanh(u)
        logp_per_dim = dist.log_prob(u) - torch.log(1 - squashed_a.pow(2) + 1e-6)

        # Replace the old unimodal tanh-Gaussian pulse phase slice with the
        # backbone's autoregressive circular mixture.  Non-phase dimensions keep
        # SAC's squashed Gaussian; phase slots are emitted in radians so the env
        # can pass them directly to pulse_phase_rel_rad overrides.  Build the
        # final action out-of-place so autograd does not see a CopySlices version
        # bump from overwriting ``squashed_a`` after ``logp_per_dim`` captures it.
        pulse_slice = self._pulse_phase_action_slice()
        phase_mix = self.backbone.pulse_phase_autoregressive_mixture(z, deterministic=False)
        pulse_phases = phase_mix["pulse_phase_rel_rad"]
        pulse_logp, _ = self.backbone.pulse_phase_autoregressive_log_prob(z, pulse_phases)
        a = torch.cat(
            [
                squashed_a[:, :pulse_slice.start],
                pulse_phases,
                squashed_a[:, pulse_slice.stop:],
            ],
            dim=-1,
        )
        keep = torch.ones(self.action_dim, dtype=torch.bool, device=squashed_a.device)
        keep[pulse_slice] = False
        logp = logp_per_dim[:, keep].sum(dim=-1) + pulse_logp
        return a, logp


class SACCriticVarLen(torch.nn.Module):
    """SAC Q-network paired with autoregressive-mixture pulse phase actions."""

    def __init__(self, action_dim: int, in_ch: int = 14, base_ch: int = 24, n_scalar_features: int = 6, max_tones: int = 128, max_pulses: int = 128, pulse_phase_ar_components: int = 4):
        super().__init__()
        self.backbone = TonePulseTXControlNetVarLen(
            in_ch=in_ch,
            base_ch=base_ch,
            n_scalar_features=n_scalar_features,
            max_tones=max_tones,
            max_pulses=max_pulses,
            pulse_phase_ar_components=pulse_phase_ar_components,
        )
        self.q_head = torch.nn.Sequential(
            torch.nn.Linear(96 + int(action_dim), 192),
            torch.nn.ReLU(inplace=True),
            torch.nn.Linear(192, 96),
            torch.nn.ReLU(inplace=True),
            torch.nn.Linear(96, 1),
        )

    def _fused(self, obs: Dict[str, torch.Tensor]) -> torch.Tensor:
        stft_list = [
            torch.clamp(
                torch.nan_to_num(x, nan=0.0, posinf=1.0e6, neginf=-1.0e6),
                min=-1.0e6,
                max=1.0e6,
            )
            for x in obs["stft_feature_list"]
        ]
        ref = stft_list[0]
        n_scalar = self.backbone.scalar_proj[0].in_features
        scalar_side = obs.get("scalar_side", torch.zeros(ref.shape[0], n_scalar, device=ref.device, dtype=ref.dtype))
        emb = [self.backbone.encoder(x) for x in stft_list]
        z_stft = torch.stack(emb, dim=0).mean(dim=0)
        fused = self.backbone.fusion(torch.cat([z_stft, self.backbone.scalar_proj(scalar_side)], dim=-1))
        return torch.nan_to_num(fused, nan=0.0, posinf=1.0e6, neginf=-1.0e6)

    def forward(self, obs: Dict[str, torch.Tensor], action: torch.Tensor) -> torch.Tensor:
        z = self._fused(obs)
        action = torch.clamp(torch.nan_to_num(action, nan=0.0, posinf=20.0, neginf=-20.0), min=-20.0, max=20.0)
        return self.q_head(torch.cat([z, action], dim=-1))


In [17]:
# Soft Actor-Critic (SAC) training loop variant for batched jammer environment.
# This keeps the same notebook utilities while swapping in entropy-regularized RL updates.
def detach_obs(obs):
    """Return a replay-buffer-safe observation with tensors detached from autograd."""
    return {
        "stft_feature_list": [x.detach() for x in obs["stft_feature_list"]],
        **({
            "scalar_side": obs["scalar_side"].detach()
        } if "scalar_side" in obs else {}),
    }


def _env_split_pool(env, split):
    """Return the JammerVecEnv sample pool for a split when available."""
    if split == "test":
        return getattr(env, "_test_pool", None)
    return getattr(env, "_train_pool", None)


def _env_has_split(env, split):
    """Detect eager and lazy JammerVecEnv splits without relying on public list aliases."""
    split_samples = getattr(env, "test_samples", None) if split == "test" else getattr(env, "samples", None)
    if split_samples:
        return True

    split_pool = _env_split_pool(env, split)
    if split_pool is None:
        return False
    if getattr(split_pool, "lazy", False):
        # DataLoader-backed JammerVecEnv keeps env.test_samples/env.samples empty
        # intentionally, so the pool's presence is the signal that the split exists.
        return True
    return len(getattr(split_pool, "items", [])) > 0


def _env_default_eval_steps(env, split):
    """Choose a default eval length for eager lists and lazy DataLoader-backed pools."""
    split_samples = getattr(env, "test_samples", None) if split == "test" else getattr(env, "samples", None)
    if split_samples:
        return max(1, int(np.ceil(len(split_samples) / max(1, int(env.num_envs)))))

    split_pool = _env_split_pool(env, split)
    if split_pool is not None:
        try:
            known_batches = int(len(split_pool))
        except Exception:
            known_batches = 0
        if known_batches > 0:
            return known_batches
    return 1


def _reset_lazy_pool(pool):
    """Rewind a lazy _SamplePool so each evaluation starts from the split beginning."""
    if pool is not None and getattr(pool, "lazy", False):
        pool._iterator = None
        pool._buffer = []
        pool._batches_seen = 0


def _env_split_sample_count(env, split):
    """Best-effort row count for eager lists and lazy DataLoader-backed pools."""
    split_samples = getattr(env, "test_samples", None) if split == "test" else getattr(env, "samples", None)
    if split_samples:
        return len(split_samples)

    split_pool = _env_split_pool(env, split)
    iterable = getattr(split_pool, "_iterable", None) if split_pool is not None else None
    dataset = getattr(iterable, "dataset", None)
    if dataset is not None:
        try:
            return int(len(dataset))
        except Exception:
            pass
    sampler = getattr(iterable, "sampler", None)
    if sampler is not None:
        try:
            return int(len(sampler))
        except Exception:
            pass
    return 0


def _env_split_batch_sizes(env, split, batch_size, steps=None):
    """Return per-substep env widths that cover a split without materializing it."""
    batch_size = max(1, int(batch_size))
    if steps is not None:
        return [batch_size] * max(1, int(steps))

    sample_count = _env_split_sample_count(env, split)
    if sample_count > 0:
        return [
            min(batch_size, sample_count - start)
            for start in range(0, sample_count, batch_size)
        ]

    # Fallback for iterables that expose only batch count.  This still walks the
    # whole source once while preserving bounded batch memory.
    return [batch_size] * _env_default_eval_steps(env, split)


def _rewind_env_split(env, split):
    """Reset a split cursor/pool so a pass starts from the beginning."""
    _reset_lazy_pool(_env_split_pool(env, split))
    if hasattr(env, "_cursor"):
        env._cursor[split] = 0
    env._active = []
    env._step_count = 0
    env._epoch_complete = False


@torch.no_grad()
def _evaluate_sac_split_metrics(
    *,
    env,
    actor,
    split="test",
    device="cuda",
    eval_batch_size=None,
    eval_steps=None,
    eval_batch_sizes=None,
):
    """Evaluate the SAC actor on a train/test split without perturbing env state."""
    split_pool = _env_split_pool(env, split)
    if not _env_has_split(env, split):
        return {
            "reward_mean": float("inf"),
            "decode_pct": 0.0,
            "decode_successes": 0,
            "decode_total": 0,
            "steps": 0,
        }

    original_mode = getattr(env, "mode", "train")
    original_active = list(getattr(env, "_active", []))
    original_step_count = int(getattr(env, "_step_count", 0))
    original_cursor = dict(getattr(env, "_cursor", {}))
    original_epoch_complete = bool(getattr(env, "_epoch_complete", False))
    original_num_envs = int(getattr(env, "num_envs", 1))
    original_max_steps = int(getattr(env, "max_steps", 1))
    actor_was_training = actor.training

    rewards_sum = 0.0
    rewards_count = 0
    decode_successes = 0
    decode_total = 0

    try:
        actor.eval()
        env.set_mode(split)
        _rewind_env_split(env, split)
        # Use the same full-split sweep logic as training: evaluate in bounded
        # sub-batches and size the final sub-batch to the remaining test rows.
        # ``eval_steps`` remains an explicit override for callers that want to
        # cap evaluation length, but the default covers the entire split.
        if eval_batch_sizes is None:
            eval_step_batch_size = original_num_envs if eval_batch_size is None else int(eval_batch_size)
            eval_batch_sizes = _env_split_batch_sizes(
                env,
                split,
                eval_step_batch_size,
                steps=eval_steps,
            )
        else:
            eval_batch_sizes = [int(size) for size in eval_batch_sizes]

        # Prevent JammerVecEnv.step from prefetching the next batch.  We advance
        # the split explicitly below so partial final batches are not padded with
        # rows from the next epoch.
        env.max_steps = max(original_max_steps, 2)
        test_batch = 0
        for current_batch_size in eval_batch_sizes:
            print(f'test batch : {test_batch}')
            test_batch += 1
            env.num_envs = int(current_batch_size)
            env._step_count = 0
            env._active = env._next_samples()
            active_samples = list(env._active)
            iq_batches = stack_iq_views_to_device(active_samples, device=device)
            model_obs = build_stft_observation_fast(
                iq1=iq_batches[0],
                iq2=iq_batches[1],
                iq3=iq_batches[2],
                intake_sample_rate_hz=float(env.jammer_sampling_freq),
                device=device,
            )
            action_t, _ = actor.sample(model_obs)
            jam_batch = jammer_controller_batch(
                model=env.model,
                samples=active_samples,
                actions=action_t.detach(),
                jammer_sampling_freq=env.jammer_sampling_freq,
                default_output_len = env.default_output_len,
                user_peak_power_fraction=env.user_peak_power_fraction,
                device=device,
                rx_iq_batches=iq_batches,
                action_cfg={},
            )
            rewards_t, step_decodes, step_total = env.reward_fn(jam_batch, active_samples)
            rewards_t = torch.as_tensor(rewards_t, device=device, dtype=torch.float32)
            env._step_count += 1
            rewards_sum += float(rewards_t.sum().detach().item())
            rewards_count += int(rewards_t.numel())
            decode_successes += int(step_decodes)
            decode_total += int(step_total)
            env._active = []
            del model_obs, jam_batch
            clean()


        reward_mean = (rewards_sum / rewards_count) if rewards_count else float("inf")
        decode_pct = (100.0 * decode_successes / decode_total) if decode_total else 0.0
        return {
            "reward_mean": float(reward_mean),
            "decode_pct": float(decode_pct),
            "decode_successes": int(decode_successes),
            "decode_total": int(decode_total),
            "steps": int(len(eval_batch_sizes)),
        }
    finally:
        _reset_lazy_pool(split_pool)
        env.num_envs = original_num_envs
        env.max_steps = original_max_steps
        if hasattr(env, "set_mode"):
            env.set_mode(original_mode)
        env._active = original_active
        env._step_count = original_step_count
        if original_cursor:
            env._cursor = original_cursor
        env._epoch_complete = original_epoch_complete
        actor.train(actor_was_training)


def train_rl_batched(
    env,
    actor,
    critic1,
    critic2,
    target_critic1,
    target_critic2,
    actor_optimizer,
    critic1_optimizer,
    critic2_optimizer,
    action_dim,
    log_alpha,
    alpha_optimizer,
    replay_buffer,
    reward_sched=None,
    device="cuda",
    gamma=0.99,
    tau=0.005,
    target_entropy=-8.0,
    batch_size=250,
    updates=100,
    warmup_steps=200,
    tensorboard_log_dir="runs/train_rl_batched_sac",
    log_interval=50,
    enable_tensorboard=True,
    test_eval_interval=None,
    test_eval_steps=None,
    checkpoint_dir="checkpoints_rl/checkpoints_rl_sac",
    checkpoint_name="best_model_sac.pt",
    enable_checkpointing=True,
    checkpoint_mode="max",
):
    """Train with Soft Actor-Critic over the existing batched jammer env.

    Notes:
      - Assumes actor returns (action, log_prob) through actor.sample(obs).
      - Assumes critic modules take (obs, action) and return Q-values.
      - `batch_size` is used as the maximum environment sub-batch width and as
        the replay minibatch size.  Each outer update sweeps the whole train
        split in sub-batches so memory stays bounded without skipping rows.
      - `replay_buffer` exposes add(...) and sample(batch_size).
      - When `enable_tensorboard` is true, SAC losses, rewards, alpha, and
        decode percentage are written to `tensorboard_log_dir`.
      - If `env.test_samples` is populated (for example, by constructing
        `JammerVecEnv(..., test_samples=test_loader, ...)`), the actor is
        evaluated on the full test split every `test_eval_interval` updates
        using the same bounded sub-batch sweep as training, and the best
        checkpoint is saved using the selected `checkpoint_mode`.
    """
    writer = SummaryWriter(log_dir=tensorboard_log_dir) if enable_tensorboard else None
    has_test_split = _env_has_split(env, "test")
    if test_eval_interval is None:
        test_eval_interval = log_interval
    test_eval_interval = max(1, int(test_eval_interval))
    checkpoint_mode = str(checkpoint_mode).lower()
    if checkpoint_mode not in ("min", "max"):
        raise ValueError("checkpoint_mode must be 'min' or 'max'")
    best_test_metric = float("inf") if checkpoint_mode == "min" else -float("inf")
    best_checkpoint_path = None
    if enable_checkpointing and has_test_split:
        checkpoint_dir = Path(checkpoint_dir)
        checkpoint_dir.mkdir(parents=True, exist_ok=True)
        best_checkpoint_path = checkpoint_dir / checkpoint_name

    # original_num_envs = int(getattr(env, "num_envs", 1))
    train_sample_count = len(getattr(env, "samples", []))
    env_step_batch_size = max(1, int(batch_size))
    if train_sample_count > 0:
        env_step_batch_size = min(env_step_batch_size, train_sample_count)

    original_num_envs = int(getattr(env, "num_envs", 1))
    original_max_steps = int(getattr(env, "max_steps", 1))
    if hasattr(env, "set_mode"):
        env.set_mode("train")
    train_batch_sizes = _env_split_batch_sizes(env, "train", env_step_batch_size)
    test_batch_sizes = (
        _env_split_batch_sizes(env, "test", env_step_batch_size, steps=test_eval_steps)
        if has_test_split
        else []
    )
    reward_state = {"step_idx": 0}
    original_reward_fn = getattr(env, "reward_fn", None)
    if reward_sched is not None:
        env.reward_fn = lambda jam_batch, samples: compute_rewards_scheduled_batched(
            jam_batch=jam_batch,
            samples=samples,
            jammer_sampling_freq=float(env.jammer_sampling_freq),
            device=device,
            step_idx=int(reward_state["step_idx"]),
            sched=reward_sched,
        )
    global_step = 0
    decode_successes = 0
    decode_total = 0
    reward_ema = None
    train_decode_pct_ema = None
    test_decode_pct_ema = None
    ema_beta = 0.9
    last_critic1_loss = None
    last_critic2_loss = None
    last_actor_loss = None
    last_alpha_loss = None
    latest_test_metrics = None

    try:
        env.max_steps = max(original_max_steps, 2)
        for step in range(updates):
            reward_state["step_idx"] = step
            if hasattr(env, "set_mode"):
                env.set_mode("train")
            _rewind_env_split(env, "train")

            step_reward_sum = 0.0
            step_reward_count = 0
            step_decodes = 0
            step_total = 0
            step_logp_sum = 0.0
            step_logp_count = 0
            batch = 0

            for current_batch_size in train_batch_sizes:
                print(f'batch : {batch}')
                batch += 1
                env.num_envs = int(current_batch_size)
                env._step_count = 0
                env._active = env._next_samples()
                active_samples = list(env._active)
                iq_batches = stack_iq_views_to_device(active_samples, device=device)
                model_obs = build_stft_observation_fast(
                    iq1=iq_batches[0],
                    iq2=iq_batches[1],
                    iq3=iq_batches[2],
                    intake_sample_rate_hz=float(env.jammer_sampling_freq),
                    device=device,
                )

                with torch.no_grad():
                    if step < warmup_steps:
                        # Warmup: random exploratory actions before bootstrapped SAC updates.
                        action_t = torch.rand((env.num_envs, action_dim), device=device) * 2.0 - 1.0
                        logp_t = torch.zeros(env.num_envs, device=device)
                    else:
                        action_t, logp_t = actor.sample(model_obs)

                jam_batch = jammer_controller_batch(
                    model=env.model,
                    samples=active_samples,
                    actions=action_t.detach(),
                    jammer_sampling_freq=env.jammer_sampling_freq,
                    default_output_len=env.default_output_len,
                    user_peak_power_fraction=env.user_peak_power_fraction,
                    device=device,
                    rx_iq_batches=iq_batches,
                    action_cfg={},
                )
                rewards_t, batch_decodes, batch_total = env.reward_fn(jam_batch, active_samples)
                rewards_t = torch.as_tensor(rewards_t, device=device, dtype=torch.float32).unsqueeze(-1)
                rewards_t = torch.nan_to_num(rewards_t, nan=0.0, posinf=1.0e6, neginf=-1.0e6)
                env._step_count += 1

                replay_buffer.add(
                    detach_obs(model_obs),
                    action_t.detach(),
                    rewards_t,
                )

                batch_reward_sum = float(rewards_t.sum().detach().item())
                batch_reward_count = int(rewards_t.numel())
                step_reward_sum += batch_reward_sum
                step_reward_count += batch_reward_count
                step_decodes += int(batch_decodes)
                step_total += int(batch_total)
                step_logp_sum += float(logp_t.sum().detach().item())
                step_logp_count += int(logp_t.numel())
                decode_successes += int(batch_decodes)
                decode_total += int(batch_total)
                global_step += env.num_envs
                env._active = []
                del model_obs, jam_batch
                clean()

            step_decode_pct = (100.0 * step_decodes / step_total) if step_total else 0.0
            reward_mean = (step_reward_sum / step_reward_count) if step_reward_count else float("inf")
            reward_ema = reward_mean if reward_ema is None else (ema_beta * reward_ema + (1.0 - ema_beta) * reward_mean)
            train_decode_pct_ema = step_decode_pct if train_decode_pct_ema is None else (ema_beta * train_decode_pct_ema + (1.0 - ema_beta) * step_decode_pct)
            mean_logp = (step_logp_sum / step_logp_count) if step_logp_count else 0.0

            did_update = step >= warmup_steps and len(replay_buffer) >= batch_size
            if did_update:
                print('updating')
                s, a, r = replay_buffer.sample(batch_size)

                # One-step terminal rollout: no next-state bootstrap is used.
                y = r

                q1 = critic1(s, a)
                q2 = critic2(s, a)
                critic1_loss = torch.nn.functional.mse_loss(q1, y)
                critic2_loss = torch.nn.functional.mse_loss(q2, y)

                critic1_optimizer.zero_grad(set_to_none=True)
                critic1_loss.backward()
                torch.nn.utils.clip_grad_norm_(critic1.parameters(), max_norm=10.0)
                critic1_optimizer.step()

                critic2_optimizer.zero_grad(set_to_none=True)
                critic2_loss.backward()
                torch.nn.utils.clip_grad_norm_(critic2.parameters(), max_norm=10.0)
                critic2_optimizer.step()

                a_pi, logp_pi = actor.sample(s)
                q1_pi = critic1(s, a_pi)
                q2_pi = critic2(s, a_pi)
                q_pi = torch.min(q1_pi, q2_pi)
                alpha = log_alpha.exp()

                # SAC actor objective:
                # J_pi = E[ alpha*log_pi(a|s) - Q(s,a) ]
                # The entropy term alpha*log_pi encourages exploration and smoother policies.
                actor_loss = (alpha * logp_pi.unsqueeze(-1) - q_pi).mean()

                actor_optimizer.zero_grad(set_to_none=True)
                actor_loss.backward()
                torch.nn.utils.clip_grad_norm_(actor.parameters(), max_norm=10.0)
                actor_optimizer.step()

                # Temperature (entropy weight) objective:
                # J_alpha = E[ -alpha * (log_pi(a|s) + target_entropy) ]
                # If policy entropy is too low, this increases alpha; if too high, alpha decreases.
                alpha_loss = -(log_alpha * (logp_pi.detach() + target_entropy)).mean()
                alpha_optimizer.zero_grad(set_to_none=True)
                alpha_loss.backward()
                alpha_optimizer.step()

                last_critic1_loss = float(critic1_loss.detach().item())
                last_critic2_loss = float(critic2_loss.detach().item())
                last_actor_loss = float(actor_loss.detach().item())
                last_alpha_loss = float(alpha_loss.detach().item())

            should_eval_test = has_test_split and ((step + 1) % test_eval_interval == 0 or (step + 1) == updates)
            if should_eval_test and ((step + 1) >= warmup_steps):
                print(f'testing, step: {step + 1}')
                latest_test_metrics = _evaluate_sac_split_metrics(
                    env=env,
                    actor=actor,
                    split="test",
                    device=device,
                    eval_batch_size=env_step_batch_size,
                    eval_steps=test_eval_steps,
                    eval_batch_sizes=test_batch_sizes,
                )
                test_decode_pct = float(latest_test_metrics["decode_pct"])
                test_decode_pct_ema = test_decode_pct if test_decode_pct_ema is None else (ema_beta * test_decode_pct_ema + (1.0 - ema_beta) * test_decode_pct)
                latest_test_metrics["decode_pct_ema"] = test_decode_pct_ema
                test_metric = float(latest_test_metrics["reward_mean"])
                is_best = test_metric < best_test_metric if checkpoint_mode == "min" else test_metric > best_test_metric
                if is_best:
                    best_test_metric = test_metric
                    if best_checkpoint_path is not None:
                        print(f"saving best SAC model at step {step + 1}")
                        torch.save(
                            {
                                "step": step + 1,
                                "global_step": global_step,
                                "actor_state_dict": actor.state_dict(),
                                "critic1_state_dict": critic1.state_dict(),
                                "critic2_state_dict": critic2.state_dict(),
                                "target_critic1_state_dict": target_critic1.state_dict(),
                                "target_critic2_state_dict": target_critic2.state_dict(),
                                "actor_optimizer_state_dict": actor_optimizer.state_dict(),
                                "critic1_optimizer_state_dict": critic1_optimizer.state_dict(),
                                "critic2_optimizer_state_dict": critic2_optimizer.state_dict(),
                                "alpha_optimizer_state_dict": alpha_optimizer.state_dict(),
                                "log_alpha": log_alpha.detach().cpu(),
                                "alpha": log_alpha.exp().detach().cpu(),
                                "train_reward_step": reward_mean,
                                "train_reward_ema": reward_ema,
                                "train_decode_pct": step_decode_pct,
                                "train_decode_pct_ema": train_decode_pct_ema,
                                "test_reward_mean": test_metric,
                                "test_decoded_pct": float(latest_test_metrics["decode_pct"]),
                                "test_decoded_pct_ema": float(latest_test_metrics["decode_pct_ema"]),
                                "test_decode_successes": int(latest_test_metrics["decode_successes"]),
                                "test_decode_total": int(latest_test_metrics["decode_total"]),
                                "checkpoint_mode": checkpoint_mode,
                            },
                            best_checkpoint_path,
                        )

            if writer is not None:
                writer.add_scalar("train/reward_step", reward_mean, step)
                writer.add_scalar("train/reward_ema", reward_ema, step)
                writer.add_scalar("train/decode_pct", step_decode_pct, step)
                writer.add_scalar("train/decode_pct_ema", train_decode_pct_ema, step)
                writer.add_scalar("train/decode_successes", step_decodes, step)
                writer.add_scalar("train/decode_total", step_total, step)
                writer.add_scalar("train/replay_size", len(replay_buffer), step)
                writer.add_scalar("train/alpha", log_alpha.exp().detach().item(), step)
                writer.add_scalar("train/action_logp", float(mean_logp), step)
                if latest_test_metrics is not None:
                    writer.add_scalar("test/reward_mean", float(latest_test_metrics["reward_mean"]), step)
                    writer.add_scalar("test/decoded_pct", float(latest_test_metrics["decode_pct"]), step)
                    writer.add_scalar("test/decode_pct_ema", float(latest_test_metrics["decode_pct_ema"]), step)
                    writer.add_scalar("test/decode_successes", int(latest_test_metrics["decode_successes"]), step)
                    writer.add_scalar("test/decode_total", int(latest_test_metrics["decode_total"]), step)
                    writer.add_scalar("test/best_reward_mean", best_test_metric, step)
                if did_update:
                    writer.add_scalar("loss/critic1", last_critic1_loss, step)
                    writer.add_scalar("loss/critic2", last_critic2_loss, step)
                    writer.add_scalar("loss/actor", last_actor_loss, step)
                    writer.add_scalar("loss/alpha", last_alpha_loss, step)

            if (step + 1) % log_interval == 0:
                loss_msg = ""
                if last_critic1_loss is not None:
                    loss_msg = (
                        f" c1={last_critic1_loss:.4f} c2={last_critic2_loss:.4f} "
                        f"actor={last_actor_loss:.4f}"
                    )
                test_msg = ""
                if latest_test_metrics is not None:
                    test_msg = (
                        f" test_reward={latest_test_metrics['reward_mean']:.4f} "
                        f"test_decode_pct={latest_test_metrics['decode_pct']:.2f} "
                        f"test_decode_pct_ema={latest_test_metrics['decode_pct_ema']:.2f}"
                    )
                log_message = (
                    f"step={step+1} reward={reward_mean:.4f} reward_ema={reward_ema:.4f} "
                    f"decode_pct={step_decode_pct:.2f} decode_pct_ema={train_decode_pct_ema:.2f} "
                    f"step_total={step_total:.2f} alpha={log_alpha.exp().item():.4f}{loss_msg}{test_msg}"
                )
                logger.info(log_message)
                print(log_message)
    finally:
        env.num_envs = original_num_envs
        env.max_steps = original_max_steps
        if reward_sched is not None and original_reward_fn is not None:
            env.reward_fn = original_reward_fn
        if writer is not None:
            writer.close()

    return {
        "actor": actor,
        "critic1": critic1,
        "critic2": critic2,
        "target_critic1": target_critic1,
        "target_critic2": target_critic2,
        "alpha": log_alpha.exp().detach().item(),
        "decode_pct": (100.0 * decode_successes / decode_total) if decode_total else 0.0,
        "decode_pct_ema": train_decode_pct_ema,
        "decode_successes": decode_successes,
        "decode_total": decode_total,
        "latest_test_metrics": latest_test_metrics,
        "best_test_metric": best_test_metric if has_test_split else None,
        "best_checkpoint_path": str(best_checkpoint_path) if best_checkpoint_path is not None else None,
        "tensorboard_log_dir": tensorboard_log_dir if enable_tensorboard else None,
    }


In [18]:
max_tones = 32
max_pulses = 1024
action_dim = 11 + 4*max_tones + max_pulses  # unchanged; pulse phase slice is sampled by an AR mixture
print(f'action_dim: {action_dim}')
actor = SACActorVarLen(
        action_dim = action_dim,
        in_ch = 14,
        base_ch = 24,
        max_tones = max_tones,
        max_pulses = max_pulses,
        pulse_phase_ar_components = 4,
    ).to("cuda" if torch.cuda.is_available() else "cpu")


# critic_max_tones = 128
# critic_max_pulses = 128
# critic_action_dim = 11 + 4*critic_max_tones + critic_max_pulses
# print(f'critic_action_dim: {critic_action_dim}')

critic_0 = SACCriticVarLen(
        action_dim = action_dim,
        in_ch = 14,
        base_ch = 12,
        max_tones = max_tones,
        max_pulses = max_pulses,
        pulse_phase_ar_components = 4,
    ).to("cuda" if torch.cuda.is_available() else "cpu")

critic_1 = SACCriticVarLen(
        action_dim = action_dim,
        in_ch = 14,
        base_ch = 12,
        max_tones = max_tones,
        max_pulses = max_pulses,
        pulse_phase_ar_components = 4,
    ).to("cuda" if torch.cuda.is_available() else "cpu")


action_dim: 1163


In [19]:
jve = JammerVecEnv(
        samples = train_loader,
        test_samples = test_loader,
        model = actor,
        jammer_sampling_freq = float(jammer_sampling_freq),
        num_envs = 1,
        reward_fn = None,
        user_peak_power_fraction= 10.0,
        default_output_len = 1_000_000,
        max_steps = 1,
        device = device)

cfg = PPOConfig()
rsc = RewardScheduleConfig()
cfg.rollout_steps = jve._source_batches_per_epoch

In [ ]:
run_no = 73
tensorboard_log_dir=f"runs/train_rl_batched_sac_{run_no}"
checkpoint_name=f"best_model_sac_{run_no}.pt"

lr = 8e-5

initial_alpha = 0.05
log_alpha = torch.tensor(
    math.log(initial_alpha),
    device=device,
    dtype=torch.float32,
    requires_grad=True,
)

alpha_optimizer = torch.optim.Adam(
    [log_alpha],
    lr=1e-4,
)

replay_buffer = SACReplayBuffer(
    capacity=500,
    device=device,
)

reward_sched = RewardScheduleConfig(decode_every_n_steps=1, proxy_reward_scale=0.01)

p = train_rl_batched(
    env = jve,
    actor = actor,
    critic1 = critic_0,
    critic2 = critic_1,
    target_critic1 = copy.deepcopy(critic_0).to('cuda'),
    target_critic2  = copy.deepcopy(critic_1).to('cuda'),
    actor_optimizer = torch.optim.Adam(actor.parameters(), lr=lr),
    critic1_optimizer = torch.optim.Adam(critic_0.parameters(), lr=lr),
    critic2_optimizer = torch.optim.Adam(critic_1.parameters(), lr=lr),
    action_dim = action_dim,
    log_alpha = log_alpha,
    alpha_optimizer = alpha_optimizer,
    replay_buffer = replay_buffer,
    reward_sched=reward_sched,
    device="cuda",
    gamma=0.99,
    tau=0.005,
    target_entropy = -4,#-0.25*float(action_dim),
    batch_size = batch_size,
    updates = 324,
    warmup_steps = 1,
    test_eval_interval = 4,
    checkpoint_dir="checkpoints_rl/checkpoints_rl_sac",
    checkpoint_name=checkpoint_name,
    tensorboard_log_dir=tensorboard_log_dir,
    log_interval = 1
)

batch : 0
batch : 1
batch : 2
batch : 3
batch : 4
batch : 5
batch : 6
batch : 7
batch : 8
batch : 9
batch : 10
batch : 11
batch : 12
batch : 13
batch : 14
batch : 15
batch : 16
batch : 17
batch : 18
